# EDA — multi-label damage segmentation (unified 5-class schema)

Presentation layer over `scripts/run_eda.py` (single source of truth for all numbers). Re-generate the data first:

```bash
python scripts/run_eda.py --config configs/experiments/baseline_smp_efficientnet_b0.yaml --output_dir docs/eda
```

Run this notebook from the repository root (or any subfolder — the root is found automatically). Physical sizes (mm) are not available: the source GSD is unknown.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'schema.py').exists())
sys.path.insert(0, str(ROOT))
S = json.loads((ROOT / 'docs/eda/eda_summary.json').read_text())
C = S['classes']
print('classes:', C, '| dacl10k strategy:', S['dacl10k_test_strategy'])
print({k: v['n_images'] for k, v in S['splits'].items()})

## 1. Class frequencies — instances, image share, pixel share

In [ ]:
rows = []
for k, s in S['splits'].items():
    for c in C:
        rows.append(dict(split=k, cls=c, instances=s['instances'][c], pct_images=100*s['image_fraction_with_class'][c], pct_pixels=100*s['pixel_share_of_all_pixels'][c]))
df = pd.DataFrame(rows)
display(df.pivot(index='cls', columns='split', values='instances'))
display(df.pivot(index='cls', columns='split', values='pct_pixels').round(2))

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'docs/eda/class_frequencies.png')))

**Reading:** instance counts are not rare, but pixel shares differ by ~10x between sources — cif_coco 'Cracks' are large region labels (17.7% of pixels), dacl10k cracks are thin lines (1.8%).

## 2. Instance size distribution (per class)

Characteristic size = sqrt(area) at the 512 px evaluation resolution; also image-area fractions and bbox aspect.

In [ ]:
rows = []
for k, s in S['splits'].items():
    if not k.endswith('/train'): continue
    for c in C:
        rows.append(dict(split=k, cls=c, **{f'size512_{q}': round(v, 1) for q, v in s['instance_char_size_at_eval_res_px'][c].items()}, aspect_p50=round(s['instance_bbox_aspect'][c]['p50'], 2)))
display(pd.DataFrame(rows))
display(Image(filename=str(ROOT / 'docs/eda/instance_sizes.png')))

## 3. Overlap / co-occurrence (why the task is multi-label)

P(column | row) at pixel level. Overlap is essentially a dacl10k phenomenon.

In [ ]:
for k in [k for k in S['splits'] if k.endswith('/train')]:
    M = pd.DataFrame(S['splits'][k]['pixel_cooccurrence_P_b_given_a'], index=C, columns=C).round(2)
    print(k); display(M)
display(Image(filename=str(ROOT / 'docs/eda/cooccurrence.png')))

## 4. Image statistics and resolutions

In [ ]:
for name, st in S.get('image_stats_train', {}).items():
    print(name, {k: v for k, v in st.items()})
for k, s in S['splits'].items():
    if k.endswith('/train'): print(k, 'distinct resolutions:', s['n_distinct_resolutions'], '| most common:', s['resolutions'])

## 5. Sample visualisations (annotation overlays under the unified taxonomy)

Note the black padding in cif_coco tiles — excluded from loss and metrics by the validity mask.

In [ ]:
for name in ['cif_coco', 'dacl10k']:
    print(name); display(Image(filename=str(ROOT / f'docs/eda/samples_{name}.png')))

## 6. No-data padding (validity mask) — measured on a random sample of train images

In [ ]:
from configs.loader import load_experiment_config
from data.datasets.combined_dataset import build_split_datasets
from data.validity_mask import no_data_validity_mask
cfg = load_experiment_config(ROOT / 'configs/experiments/baseline_smp_efficientnet_b0.yaml')
ds = build_split_datasets(cfg.data, None, None)['train']
rng = np.random.default_rng(0)
for d in ds.datasets:
    fr = []
    for i in rng.choice(len(d), 100, replace=False):
        img = d._load_image(d.images_by_id[d.image_ids[i]]['file_name'])
        fr.append(1 - no_data_validity_mask(img, max_pixels=2_000_000).mean())
    fr = np.array(fr)
    print(f'{d.source_name}: mean flagged no-data pixels {100*fr.mean():.1f}% | images with >1% flagged {100*(fr>0.01).mean():.0f}% | >20% {100*(fr>0.2).mean():.0f}%')

## Takeaways

1. Multi-label overlap is real in dacl10k (e.g. P(Wetspot | Cracks) = 0.38), almost absent in cif_coco.
2. 'Cracks' has two label semantics (region vs line) — report per-source metrics.
3. Objects span 16 px (Spalling/Rust) to 146 px (Wetspot) at 512 px; cracks are thin.
4. cif_coco contains substantial black no-data padding — handled by the validity mask.
5. cif_coco is a class-balanced subset without background-only tiles (see `docs/dataset_report.md`).